# Homework 10: Multimodal Search — Text-to-Image

## Track A: Image Search by Text Query

**Goal:** Build a multimodal search system that finds relevant images based on text queries using CLIP embeddings and a vector database.

### Pipeline:
1. **Data Preparation** — Load Conceptual Captions dataset, download images, create CLIP embeddings
2. **Index Creation** — Build ChromaDB vector index over image embeddings
3. **Search Implementation** — Text-to-image search with ranking
4. **Evaluation & Analysis** — Compare CLIP variants, measure retrieval quality

### Models:
- **CLIP ViT-B/32** (`openai/clip-vit-base-patch32`) — baseline
- **OpenCLIP ViT-B/32** (`laion2b_s34b_b79k`) — comparison model

### Dataset:
- [Conceptual Captions](https://huggingface.co/datasets/google-research-datasets/conceptual_captions) — using a 1K sample for speed

## 1. Setup & Imports

In [ ]:
!pip install torch torchvision torchaudio transformers open-clip-torch chromadb datasets scikit-learn matplotlib pandas numpy pillow tqdm ftfy -q

In [ ]:
import os
import time
import warnings
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from io import BytesIO
from pathlib import Path
from tqdm import tqdm

import torch
from transformers import CLIPProcessor, CLIPModel
import open_clip

import chromadb
from chromadb.config import Settings

from datasets import load_dataset
from sklearn.metrics.pairwise import cosine_similarity

import requests
from concurrent.futures import ThreadPoolExecutor, as_completed

warnings.filterwarnings('ignore')

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
print(f"PyTorch version: {torch.__version__}")

## 2. Data Preparation

Load the Conceptual Captions dataset and download a sample of images.

In [ ]:
# Load dataset in streaming mode and take a sample
print("Loading Conceptual Captions dataset (streaming)...")
dataset = load_dataset(
    "google-research-datasets/conceptual_captions",
    split="train",
    streaming=True
)

# Take a sample of 2000 candidates (we'll try to download them, aiming for ~1000 successful)
NUM_CANDIDATES = 2000
TARGET_IMAGES = 1000

samples = []
for i, sample in enumerate(dataset):
    if i >= NUM_CANDIDATES:
        break
    samples.append(sample)

print(f"Fetched {len(samples)} sample metadata entries")
print(f"Sample entry keys: {list(samples[0].keys())}")
print(f"Example caption: {samples[0]['caption']}")
print(f"Example URL: {samples[0]['image_url']}")

In [ ]:
# Download images in parallel with timeout handling
IMAGE_DIR = Path("images")
IMAGE_DIR.mkdir(exist_ok=True)


def download_image(args):
    """Download a single image, return (index, caption, filepath) or None on failure."""
    idx, sample = args
    url = sample["image_url"]
    caption = sample["caption"]
    filepath = IMAGE_DIR / f"{idx:05d}.jpg"
    
    if filepath.exists():
        try:
            img = Image.open(filepath)
            img.verify()
            return (idx, caption, str(filepath))
        except Exception:
            filepath.unlink(missing_ok=True)
    
    try:
        resp = requests.get(url, timeout=5, stream=True)
        resp.raise_for_status()
        content_type = resp.headers.get("content-type", "")
        if "image" not in content_type:
            return None
        
        img_data = resp.content
        img = Image.open(BytesIO(img_data))
        img = img.convert("RGB")
        
        # Skip very small images
        if img.width < 64 or img.height < 64:
            return None
        
        img.save(filepath, "JPEG", quality=85)
        return (idx, caption, str(filepath))
    except Exception:
        return None


print(f"Downloading images (target: {TARGET_IMAGES})...")
successful = []

with ThreadPoolExecutor(max_workers=20) as executor:
    futures = {executor.submit(download_image, (i, s)): i for i, s in enumerate(samples)}
    pbar = tqdm(total=TARGET_IMAGES, desc="Downloaded")
    
    for future in as_completed(futures):
        result = future.result()
        if result is not None:
            successful.append(result)
            pbar.update(1)
        if len(successful) >= TARGET_IMAGES:
            # Cancel remaining futures
            for f in futures:
                f.cancel()
            break
    pbar.close()

# Sort by index to maintain consistent ordering
successful.sort(key=lambda x: x[0])
successful = successful[:TARGET_IMAGES]

# Create a DataFrame
data_df = pd.DataFrame(successful, columns=["orig_idx", "caption", "filepath"])
data_df = data_df.reset_index(drop=True)
data_df["id"] = [f"img_{i:04d}" for i in range(len(data_df))]

print(f"\nSuccessfully downloaded {len(data_df)} images")
data_df.head(10)

In [ ]:
# Visualize some sample images
fig, axes = plt.subplots(2, 5, figsize=(18, 8))
for i, ax in enumerate(axes.flat):
    if i < len(data_df):
        img = Image.open(data_df.iloc[i]["filepath"])
        ax.imshow(img)
        caption = data_df.iloc[i]["caption"]
        ax.set_title(caption[:50] + ("..." if len(caption) > 50 else ""), fontsize=8)
    ax.axis("off")
plt.suptitle("Sample Images from Dataset", fontsize=14)
plt.tight_layout()
plt.savefig("sample_images.png", dpi=100, bbox_inches="tight")
plt.show()

## 3. Create Embeddings

### 3.1 CLIP ViT-B/32 (OpenAI) — Baseline Model

In [ ]:
# Load CLIP model (OpenAI)
print("Loading CLIP ViT-B/32 (OpenAI)...")
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(DEVICE)
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
clip_model.eval()
print("CLIP model loaded.")

In [ ]:
def compute_clip_image_embeddings(filepaths, model, processor, batch_size=32):
    """Compute CLIP image embeddings in batches."""
    all_embeddings = []
    valid_indices = []
    
    for start in tqdm(range(0, len(filepaths), batch_size), desc="Computing image embeddings"):
        batch_paths = filepaths[start:start + batch_size]
        images = []
        batch_valid = []
        
        for j, fp in enumerate(batch_paths):
            try:
                img = Image.open(fp).convert("RGB")
                images.append(img)
                batch_valid.append(start + j)
            except Exception:
                continue
        
        if not images:
            continue
        
        inputs = processor(images=images, return_tensors="pt", padding=True).to(DEVICE)
        with torch.no_grad():
            embs = model.get_image_features(**inputs)
            embs = embs / embs.norm(dim=-1, keepdim=True)  # L2 normalize
        
        all_embeddings.append(embs.cpu().numpy())
        valid_indices.extend(batch_valid)
    
    return np.vstack(all_embeddings), valid_indices


def compute_clip_text_embeddings(texts, model, processor):
    """Compute CLIP text embeddings."""
    inputs = processor(text=texts, return_tensors="pt", padding=True, truncation=True).to(DEVICE)
    with torch.no_grad():
        embs = model.get_text_features(**inputs)
        embs = embs / embs.norm(dim=-1, keepdim=True)
    return embs.cpu().numpy()


# Compute image embeddings with CLIP
print("Computing CLIP image embeddings...")
t0 = time.time()
clip_image_embeddings, valid_indices = compute_clip_image_embeddings(
    data_df["filepath"].tolist(), clip_model, clip_processor, batch_size=32
)
clip_embed_time = time.time() - t0

# Filter dataframe to valid indices only
data_df = data_df.iloc[valid_indices].reset_index(drop=True)
data_df["id"] = [f"img_{i:04d}" for i in range(len(data_df))]

print(f"Generated {clip_image_embeddings.shape[0]} embeddings of dimension {clip_image_embeddings.shape[1]}")
print(f"Embedding time: {clip_embed_time:.1f}s ({clip_embed_time/len(data_df)*1000:.1f}ms per image)")

### 3.2 OpenCLIP ViT-B/32 (LAION) — Comparison Model

In [ ]:
# Load OpenCLIP model
print("Loading OpenCLIP ViT-B/32 (LAION-2B)...")
openclip_model, _, openclip_preprocess = open_clip.create_model_and_transforms(
    "ViT-B-32", pretrained="laion2b_s34b_b79k", device=DEVICE
)
openclip_tokenizer = open_clip.get_tokenizer("ViT-B-32")
openclip_model.eval()
print("OpenCLIP model loaded.")

In [ ]:
def compute_openclip_image_embeddings(filepaths, model, preprocess, batch_size=32):
    """Compute OpenCLIP image embeddings in batches."""
    all_embeddings = []
    
    for start in tqdm(range(0, len(filepaths), batch_size), desc="Computing OpenCLIP embeddings"):
        batch_paths = filepaths[start:start + batch_size]
        images = []
        
        for fp in batch_paths:
            try:
                img = Image.open(fp).convert("RGB")
                images.append(preprocess(img))
            except Exception:
                # Use a blank image as placeholder (should not happen since we filtered)
                images.append(preprocess(Image.new("RGB", (224, 224))))
        
        batch_tensor = torch.stack(images).to(DEVICE)
        with torch.no_grad():
            embs = model.encode_image(batch_tensor)
            embs = embs / embs.norm(dim=-1, keepdim=True)
        
        all_embeddings.append(embs.cpu().numpy())
    
    return np.vstack(all_embeddings)


def compute_openclip_text_embeddings(texts, model, tokenizer):
    """Compute OpenCLIP text embeddings."""
    tokens = tokenizer(texts).to(DEVICE)
    with torch.no_grad():
        embs = model.encode_text(tokens)
        embs = embs / embs.norm(dim=-1, keepdim=True)
    return embs.cpu().numpy()


# Compute OpenCLIP image embeddings
print("Computing OpenCLIP image embeddings...")
t0 = time.time()
openclip_image_embeddings = compute_openclip_image_embeddings(
    data_df["filepath"].tolist(), openclip_model, openclip_preprocess, batch_size=32
)
openclip_embed_time = time.time() - t0

print(f"Generated {openclip_image_embeddings.shape[0]} embeddings of dimension {openclip_image_embeddings.shape[1]}")
print(f"Embedding time: {openclip_embed_time:.1f}s ({openclip_embed_time/len(data_df)*1000:.1f}ms per image)")

## 4. Build Vector Database Index (ChromaDB)

We create two separate ChromaDB collections — one for each model.

In [ ]:
# Initialize ChromaDB
chroma_client = chromadb.Client(Settings(anonymized_telemetry=False))

# Delete collections if they exist (for re-runs)
for name in ["clip_images", "openclip_images"]:
    try:
        chroma_client.delete_collection(name)
    except Exception:
        pass

# Create collections
clip_collection = chroma_client.create_collection(
    name="clip_images",
    metadata={"hnsw:space": "cosine"}
)

openclip_collection = chroma_client.create_collection(
    name="openclip_images",
    metadata={"hnsw:space": "cosine"}
)

print("ChromaDB collections created.")

In [ ]:
# Index embeddings into ChromaDB
BATCH_SIZE_DB = 500

def index_embeddings(collection, embeddings, df, batch_size=BATCH_SIZE_DB):
    """Add embeddings to a ChromaDB collection in batches."""
    n = len(df)
    for start in tqdm(range(0, n, batch_size), desc=f"Indexing {collection.name}"):
        end = min(start + batch_size, n)
        collection.add(
            ids=df["id"].iloc[start:end].tolist(),
            embeddings=embeddings[start:end].tolist(),
            metadatas=[{"caption": c, "filepath": f} for c, f in
                       zip(df["caption"].iloc[start:end], df["filepath"].iloc[start:end])],
            documents=df["caption"].iloc[start:end].tolist()
        )

print("Indexing CLIP embeddings...")
t0 = time.time()
index_embeddings(clip_collection, clip_image_embeddings, data_df)
clip_index_time = time.time() - t0
print(f"CLIP indexing time: {clip_index_time:.2f}s")

print("\nIndexing OpenCLIP embeddings...")
t0 = time.time()
index_embeddings(openclip_collection, openclip_image_embeddings, data_df)
openclip_index_time = time.time() - t0
print(f"OpenCLIP indexing time: {openclip_index_time:.2f}s")

print(f"\nCLIP collection size: {clip_collection.count()}")
print(f"OpenCLIP collection size: {openclip_collection.count()}")

## 5. Search Implementation

Implement text-to-image search using both models.

In [ ]:
def search_images_clip(query, collection, model, processor, top_k=5):
    """Search images using CLIP text embeddings."""
    t0 = time.time()
    text_emb = compute_clip_text_embeddings([query], model, processor)
    encode_time = time.time() - t0
    
    t0 = time.time()
    results = collection.query(
        query_embeddings=text_emb.tolist(),
        n_results=top_k,
        include=["metadatas", "distances", "documents"]
    )
    search_time = time.time() - t0
    
    return {
        "ids": results["ids"][0],
        "captions": results["documents"][0],
        "filepaths": [m["filepath"] for m in results["metadatas"][0]],
        "distances": results["distances"][0],
        "similarities": [1 - d for d in results["distances"][0]],  # cosine distance -> similarity
        "encode_time": encode_time,
        "search_time": search_time,
        "total_time": encode_time + search_time
    }


def search_images_openclip(query, collection, model, tokenizer, top_k=5):
    """Search images using OpenCLIP text embeddings."""
    t0 = time.time()
    text_emb = compute_openclip_text_embeddings([query], model, tokenizer)
    encode_time = time.time() - t0
    
    t0 = time.time()
    results = collection.query(
        query_embeddings=text_emb.tolist(),
        n_results=top_k,
        include=["metadatas", "distances", "documents"]
    )
    search_time = time.time() - t0
    
    return {
        "ids": results["ids"][0],
        "captions": results["documents"][0],
        "filepaths": [m["filepath"] for m in results["metadatas"][0]],
        "distances": results["distances"][0],
        "similarities": [1 - d for d in results["distances"][0]],
        "encode_time": encode_time,
        "search_time": search_time,
        "total_time": encode_time + search_time
    }


def display_results(query, results, title="Search Results"):
    """Display search results as a grid."""
    n = len(results["filepaths"])
    cols = min(n, 5)
    fig, axes = plt.subplots(1, cols, figsize=(4 * cols, 4))
    if cols == 1:
        axes = [axes]
    
    for i, ax in enumerate(axes):
        if i < n:
            try:
                img = Image.open(results["filepaths"][i])
                ax.imshow(img)
                sim = results["similarities"][i]
                caption = results["captions"][i][:40]
                ax.set_title(f"#{i+1} sim={sim:.3f}\n{caption}...", fontsize=8)
            except Exception:
                ax.text(0.5, 0.5, "Error", ha="center", va="center")
        ax.axis("off")
    
    plt.suptitle(f'{title}\nQuery: "{query}"\nSearch time: {results["total_time"]*1000:.1f}ms', fontsize=11)
    plt.tight_layout()
    plt.show()


print("Search functions defined.")

In [ ]:
# Demo: Search with various queries
test_queries = [
    "a dog on the beach",
    "a red car",
    "people playing sports",
    "a beautiful sunset over mountains",
    "food on a plate",
]

print("=" * 60)
print("CLIP ViT-B/32 (OpenAI) Search Results")
print("=" * 60)

for query in test_queries:
    results = search_images_clip(query, clip_collection, clip_model, clip_processor, top_k=5)
    display_results(query, results, title="CLIP ViT-B/32")

In [ ]:
print("=" * 60)
print("OpenCLIP ViT-B/32 (LAION-2B) Search Results")
print("=" * 60)

for query in test_queries:
    results = search_images_openclip(query, openclip_collection, openclip_model, openclip_tokenizer, top_k=5)
    display_results(query, results, title="OpenCLIP ViT-B/32 (LAION-2B)")

## 6. Evaluation & Analysis

### 6.1 Text-Image Alignment Score

Since we have captions for each image, we can measure how well the search returns images whose **captions** are semantically related to the query. We use a keyword-overlap heuristic and CLIP similarity as proxy metrics.

In [ ]:
# Create evaluation queries with associated keywords for relevance scoring
eval_queries = [
    {"query": "a dog on the beach", "keywords": ["dog", "beach", "puppy", "sand", "ocean", "sea", "canine", "shore"]},
    {"query": "a red car", "keywords": ["car", "red", "vehicle", "automobile", "sports car", "sedan"]},
    {"query": "people playing sports", "keywords": ["sport", "play", "game", "player", "team", "ball", "match", "field", "athlete"]},
    {"query": "a beautiful sunset", "keywords": ["sunset", "sun", "sky", "evening", "dusk", "orange", "golden", "horizon"]},
    {"query": "food on a plate", "keywords": ["food", "plate", "dish", "meal", "cuisine", "restaurant", "eat", "cook"]},
    {"query": "a cat sitting on a couch", "keywords": ["cat", "couch", "sofa", "kitten", "sitting", "feline", "pet"]},
    {"query": "snowy mountains", "keywords": ["snow", "mountain", "peak", "winter", "alpine", "ice", "cold"]},
    {"query": "city skyline at night", "keywords": ["city", "skyline", "night", "building", "urban", "lights", "downtown"]},
    {"query": "flowers in a garden", "keywords": ["flower", "garden", "plant", "bloom", "rose", "petal", "floral"]},
    {"query": "a person reading a book", "keywords": ["read", "book", "person", "library", "study", "page"]},
]


def compute_keyword_relevance(caption, keywords):
    """Compute keyword-based relevance score (0 to 1)."""
    caption_lower = caption.lower()
    matches = sum(1 for kw in keywords if kw.lower() in caption_lower)
    return matches / len(keywords) if keywords else 0


def evaluate_search(search_fn, queries, top_k=10):
    """Evaluate search quality across multiple queries."""
    results_list = []
    
    for q_info in queries:
        query = q_info["query"]
        keywords = q_info["keywords"]
        
        results = search_fn(query, top_k=top_k)
        
        # Compute relevance for each result
        relevances = [compute_keyword_relevance(cap, keywords) for cap in results["captions"]]
        
        # Metrics
        avg_relevance = np.mean(relevances) if relevances else 0
        top1_relevance = relevances[0] if relevances else 0
        top3_relevance = np.mean(relevances[:3]) if len(relevances) >= 3 else np.mean(relevances)
        avg_similarity = np.mean(results["similarities"])
        has_relevant = int(any(r > 0 for r in relevances))  # At least one relevant result
        
        # MRR (Mean Reciprocal Rank)
        mrr = 0
        for i, r in enumerate(relevances):
            if r > 0:
                mrr = 1 / (i + 1)
                break
        
        # Precision@k (considering result relevant if keyword overlap > 0)
        p_at_5 = sum(1 for r in relevances[:5] if r > 0) / 5
        p_at_10 = sum(1 for r in relevances[:10] if r > 0) / min(10, len(relevances))
        
        results_list.append({
            "query": query,
            "avg_keyword_relevance": avg_relevance,
            "top1_relevance": top1_relevance,
            "top3_relevance": top3_relevance,
            "avg_cosine_similarity": avg_similarity,
            "has_relevant_result": has_relevant,
            "mrr": mrr,
            "p@5": p_at_5,
            "p@10": p_at_10,
            "search_time_ms": results["total_time"] * 1000
        })
    
    return pd.DataFrame(results_list)


print("Evaluation functions defined.")

In [ ]:
# Evaluate CLIP
print("Evaluating CLIP ViT-B/32 (OpenAI)...")
clip_eval = evaluate_search(
    lambda q, top_k: search_images_clip(q, clip_collection, clip_model, clip_processor, top_k),
    eval_queries,
    top_k=10
)

# Evaluate OpenCLIP
print("Evaluating OpenCLIP ViT-B/32 (LAION-2B)...")
openclip_eval = evaluate_search(
    lambda q, top_k: search_images_openclip(q, openclip_collection, openclip_model, openclip_tokenizer, top_k),
    eval_queries,
    top_k=10
)

print("\n" + "=" * 60)
print("CLIP ViT-B/32 (OpenAI) — Per-Query Results")
print("=" * 60)
print(clip_eval.to_string(index=False, float_format="%.3f"))

print("\n" + "=" * 60)
print("OpenCLIP ViT-B/32 (LAION-2B) — Per-Query Results")
print("=" * 60)
print(openclip_eval.to_string(index=False, float_format="%.3f"))

In [ ]:
# Summary comparison
def summarize_eval(eval_df, model_name):
    return {
        "Model": model_name,
        "Avg Keyword Relevance": eval_df["avg_keyword_relevance"].mean(),
        "Avg Top-1 Relevance": eval_df["top1_relevance"].mean(),
        "Avg Top-3 Relevance": eval_df["top3_relevance"].mean(),
        "Avg Cosine Similarity": eval_df["avg_cosine_similarity"].mean(),
        "MRR": eval_df["mrr"].mean(),
        "P@5": eval_df["p@5"].mean(),
        "P@10": eval_df["p@10"].mean(),
        "Hit Rate": eval_df["has_relevant_result"].mean(),
        "Avg Search Time (ms)": eval_df["search_time_ms"].mean(),
    }

summary = pd.DataFrame([
    summarize_eval(clip_eval, "CLIP ViT-B/32 (OpenAI)"),
    summarize_eval(openclip_eval, "OpenCLIP ViT-B/32 (LAION-2B)")
])

print("\n" + "=" * 60)
print("MODEL COMPARISON SUMMARY")
print("=" * 60)
print(summary.to_string(index=False, float_format="%.4f"))

In [ ]:
# Visualization: Model Comparison
metrics = ["MRR", "P@5", "P@10", "Hit Rate", "Avg Cosine Similarity"]
clip_vals = [summary.iloc[0][m] for m in metrics]
openclip_vals = [summary.iloc[1][m] for m in metrics]

x = np.arange(len(metrics))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 5))
bars1 = ax.bar(x - width/2, clip_vals, width, label="CLIP (OpenAI)", color="steelblue")
bars2 = ax.bar(x + width/2, openclip_vals, width, label="OpenCLIP (LAION-2B)", color="coral")

ax.set_ylabel("Score")
ax.set_title("Model Comparison: Retrieval Metrics")
ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.legend()
ax.set_ylim(0, 1.0)

for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax.annotate(f"{height:.3f}", xy=(bar.get_x() + bar.get_width()/2, height),
                    xytext=(0, 3), textcoords="offset points", ha="center", va="bottom", fontsize=8)

plt.tight_layout()
plt.savefig("model_comparison.png", dpi=100, bbox_inches="tight")
plt.show()

In [ ]:
# Visualization: Search time comparison
fig, ax = plt.subplots(figsize=(10, 5))

queries_short = [q["query"][:25] for q in eval_queries]
x = np.arange(len(queries_short))

ax.bar(x - width/2, clip_eval["search_time_ms"], width, label="CLIP (OpenAI)", color="steelblue")
ax.bar(x + width/2, openclip_eval["search_time_ms"], width, label="OpenCLIP (LAION-2B)", color="coral")

ax.set_ylabel("Search Time (ms)")
ax.set_title("Search Latency per Query")
ax.set_xticks(x)
ax.set_xticklabels(queries_short, rotation=45, ha="right", fontsize=8)
ax.legend()

plt.tight_layout()
plt.savefig("search_time_comparison.png", dpi=100, bbox_inches="tight")
plt.show()

In [ ]:
# Per-query relevance heatmap
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, eval_df, title in [
    (axes[0], clip_eval, "CLIP (OpenAI)"),
    (axes[1], openclip_eval, "OpenCLIP (LAION-2B)")
]:
    metric_cols = ["avg_keyword_relevance", "top1_relevance", "mrr", "p@5", "p@10"]
    data = eval_df[metric_cols].values
    
    im = ax.imshow(data, cmap="YlOrRd", aspect="auto", vmin=0, vmax=1)
    ax.set_yticks(range(len(eval_queries)))
    ax.set_yticklabels([q["query"][:20] for q in eval_queries], fontsize=8)
    ax.set_xticks(range(len(metric_cols)))
    ax.set_xticklabels(["Avg Rel", "Top-1", "MRR", "P@5", "P@10"], fontsize=8)
    ax.set_title(title)
    
    for i in range(data.shape[0]):
        for j in range(data.shape[1]):
            ax.text(j, i, f"{data[i, j]:.2f}", ha="center", va="center", fontsize=7)

plt.colorbar(im, ax=axes, shrink=0.8, label="Score")
plt.suptitle("Per-Query Retrieval Metrics Heatmap", fontsize=13)
plt.tight_layout()
plt.savefig("relevance_heatmap.png", dpi=100, bbox_inches="tight")
plt.show()

## 7. Advanced Analysis: Embedding Space Visualization

In [ ]:
from sklearn.manifold import TSNE

# Sample embeddings for t-SNE (use a subset for speed)
TSNE_SAMPLE = min(500, len(data_df))
sample_indices = np.random.choice(len(data_df), TSNE_SAMPLE, replace=False)

# Compute text embeddings for queries
query_texts = [q["query"] for q in eval_queries]
clip_query_embs = compute_clip_text_embeddings(query_texts, clip_model, clip_processor)

# Combine image and text embeddings for CLIP
combined = np.vstack([clip_image_embeddings[sample_indices], clip_query_embs])
labels = ["image"] * TSNE_SAMPLE + ["query"] * len(query_texts)

print(f"Running t-SNE on {combined.shape[0]} points ({TSNE_SAMPLE} images + {len(query_texts)} queries)...")
tsne = TSNE(n_components=2, random_state=42, perplexity=30)
tsne_result = tsne.fit_transform(combined)

# Plot
fig, ax = plt.subplots(figsize=(10, 8))

img_mask = np.array(labels) == "image"
query_mask = np.array(labels) == "query"

ax.scatter(tsne_result[img_mask, 0], tsne_result[img_mask, 1],
           c="lightblue", alpha=0.4, s=10, label="Images")
ax.scatter(tsne_result[query_mask, 0], tsne_result[query_mask, 1],
           c="red", s=100, marker="*", zorder=5, label="Queries")

for i, txt in enumerate(query_texts):
    idx = TSNE_SAMPLE + i
    ax.annotate(txt, (tsne_result[idx, 0], tsne_result[idx, 1]),
                fontsize=7, ha="left", va="bottom")

ax.set_title("t-SNE: CLIP Embedding Space (Images + Text Queries)")
ax.legend()
plt.tight_layout()
plt.savefig("tsne_embedding_space.png", dpi=100, bbox_inches="tight")
plt.show()
print("t-SNE visualization saved.")

## 8. Side-by-Side Comparison: Both Models on Same Query

In [ ]:
def side_by_side_search(query, top_k=5):
    """Run search with both models and display results side by side."""
    clip_results = search_images_clip(query, clip_collection, clip_model, clip_processor, top_k)
    openclip_results = search_images_openclip(query, openclip_collection, openclip_model, openclip_tokenizer, top_k)
    
    fig, axes = plt.subplots(2, top_k, figsize=(4 * top_k, 9))
    
    for i in range(top_k):
        # CLIP row
        try:
            img = Image.open(clip_results["filepaths"][i])
            axes[0][i].imshow(img)
            cap = clip_results["captions"][i][:35]
            sim = clip_results["similarities"][i]
            axes[0][i].set_title(f"sim={sim:.3f}\n{cap}...", fontsize=7)
        except Exception:
            axes[0][i].text(0.5, 0.5, "Error", ha="center")
        axes[0][i].axis("off")
        
        # OpenCLIP row
        try:
            img = Image.open(openclip_results["filepaths"][i])
            axes[1][i].imshow(img)
            cap = openclip_results["captions"][i][:35]
            sim = openclip_results["similarities"][i]
            axes[1][i].set_title(f"sim={sim:.3f}\n{cap}...", fontsize=7)
        except Exception:
            axes[1][i].text(0.5, 0.5, "Error", ha="center")
        axes[1][i].axis("off")
    
    axes[0][0].set_ylabel("CLIP (OpenAI)", fontsize=11, rotation=0, labelpad=80, va="center")
    axes[1][0].set_ylabel("OpenCLIP (LAION)", fontsize=11, rotation=0, labelpad=80, va="center")
    
    plt.suptitle(
        f'Query: "{query}"\n'
        f'CLIP: {clip_results["total_time"]*1000:.1f}ms | OpenCLIP: {openclip_results["total_time"]*1000:.1f}ms',
        fontsize=12
    )
    plt.tight_layout()
    plt.show()


# Run side-by-side comparisons
comparison_queries = [
    "a dog on the beach",
    "a red car",
    "people playing sports",
]

for q in comparison_queries:
    side_by_side_search(q, top_k=5)

## 9. Save Results & Summary

In [ ]:
# Save evaluation results
clip_eval.to_csv("clip_eval_results.csv", index=False)
openclip_eval.to_csv("openclip_eval_results.csv", index=False)
summary.to_csv("model_comparison_summary.csv", index=False)

# Save summary as JSON
summary_dict = {
    "dataset": {
        "name": "Conceptual Captions",
        "num_images": len(data_df),
        "embedding_dim": int(clip_image_embeddings.shape[1]),
    },
    "models": {
        "clip_vit_b32": {
            "name": "CLIP ViT-B/32 (OpenAI)",
            "embedding_time_s": round(clip_embed_time, 2),
            "index_time_s": round(clip_index_time, 2),
            "avg_search_time_ms": round(clip_eval["search_time_ms"].mean(), 2),
            "mrr": round(clip_eval["mrr"].mean(), 4),
            "p_at_5": round(clip_eval["p@5"].mean(), 4),
            "p_at_10": round(clip_eval["p@10"].mean(), 4),
            "hit_rate": round(clip_eval["has_relevant_result"].mean(), 4),
            "avg_cosine_similarity": round(clip_eval["avg_cosine_similarity"].mean(), 4),
        },
        "openclip_vit_b32": {
            "name": "OpenCLIP ViT-B/32 (LAION-2B)",
            "embedding_time_s": round(openclip_embed_time, 2),
            "index_time_s": round(openclip_index_time, 2),
            "avg_search_time_ms": round(openclip_eval["search_time_ms"].mean(), 2),
            "mrr": round(openclip_eval["mrr"].mean(), 4),
            "p_at_5": round(openclip_eval["p@5"].mean(), 4),
            "p_at_10": round(openclip_eval["p@10"].mean(), 4),
            "hit_rate": round(openclip_eval["has_relevant_result"].mean(), 4),
            "avg_cosine_similarity": round(openclip_eval["avg_cosine_similarity"].mean(), 4),
        }
    },
    "vector_db": "ChromaDB (HNSW, cosine)",
    "num_eval_queries": len(eval_queries),
}

with open("homework_10_results.json", "w") as f:
    json.dump(summary_dict, f, indent=2)

print("Results saved:")
print("  - clip_eval_results.csv")
print("  - openclip_eval_results.csv")
print("  - model_comparison_summary.csv")
print("  - homework_10_results.json")
print()
print(json.dumps(summary_dict, indent=2))

## 10. Conclusions

### Summary

In this homework, we implemented a **multimodal text-to-image search system** (Track A) with the following components:

1. **Data Preparation**: Loaded ~1000 images from the Conceptual Captions dataset with parallel downloading and validation.

2. **Embedding Models**: Compared two CLIP variants:
   - **CLIP ViT-B/32 (OpenAI)** — the original CLIP model
   - **OpenCLIP ViT-B/32 (LAION-2B)** — trained on a larger dataset (LAION-2B)

3. **Vector Index**: Built ChromaDB collections with HNSW indexing (cosine similarity) for fast approximate nearest neighbor search.

4. **Search**: Implemented text-to-image search by encoding text queries into the same embedding space as images and performing vector similarity search.

5. **Evaluation**: Measured retrieval quality using:
   - **MRR** (Mean Reciprocal Rank)
   - **Precision@K** (P@5, P@10)
   - **Hit Rate** (fraction of queries with at least one relevant result)
   - **Keyword-based relevance** (proxy for semantic relevance)
   - **Search latency**

6. **Visualization**: t-SNE embedding space visualization showing how text queries map near semantically related images.

### Key Takeaways
- CLIP models effectively encode images and text into a shared embedding space, enabling cross-modal search
- ChromaDB with HNSW provides sub-millisecond search over the collection
- Both models show reasonable retrieval quality even on a relatively small dataset
- The keyword-based evaluation is a rough proxy; in production, human relevance judgments would be preferred